In [ ]:
import sys
import os

# Google Colab / Local environment setup
if "google.colab" in sys.modules:
    print("Running in Google Colab. If you havent already, clone the repository and cd into it:")
    print("!git clone <YOUR_REPO_URL>")
    print("%cd football-shot-clustering")

# Dynamically find the project root by looking for the "src" directory
def find_project_root():
    current_dir = os.path.abspath(os.getcwd())
    while current_dir != "/":
        if os.path.exists(os.path.join(current_dir, "src", "shotquality")):
            return current_dir
        current_dir = os.path.dirname(current_dir)
    return os.path.abspath(os.getcwd()) # Fallback

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set the working directory to the project root for consistent data paths
os.chdir(PROJECT_ROOT)
print("Project root set to:", PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

from src.shotquality import config
from src.shotquality.selection import elbow_curve, suggest_k_elbow, silhouette_by_k, summarize_k_selection
from src.shotquality.clustering import run_kmeans, describe_centroids

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
# Load datasets using paths relative to PROJECT_ROOT
X_scaled = pd.read_csv("data/processed/X_scaled.csv")
X_unscaled = pd.read_csv("data/processed/X_unscaled.csv")

# Drop ID columns before passing to K-Means
cols_to_drop = [c for c in config.ID_COLS if c in X_scaled.columns]
X_train_scaled = X_scaled.drop(columns=cols_to_drop)
X_train_unscaled = X_unscaled.drop(columns=cols_to_drop)

print("Scaled shape:", X_train_scaled.shape)
print("Unscaled shape:", X_train_unscaled.shape)

In [ ]:
# 1. Elbow Method Comparison
df_elbow_scaled = elbow_curve(X_train_scaled)
best_k_elbow_scaled = suggest_k_elbow(df_elbow_scaled)

df_elbow_unscaled = elbow_curve(X_train_unscaled)
best_k_elbow_unscaled = suggest_k_elbow(df_elbow_unscaled)

fig, axes = plt.subplots(1, 2)
axes[0].plot(df_elbow_scaled["k"], df_elbow_scaled["inertia"], marker="o", linestyle="-")
axes[0].axvline(best_k_elbow_scaled, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_scaled}")
axes[0].set_title("Elbow Curve (SCALED)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].legend()

axes[1].plot(df_elbow_unscaled["k"], df_elbow_unscaled["inertia"], marker="o", linestyle="-", color="orange")
axes[1].axvline(best_k_elbow_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_unscaled}")
axes[1].set_title("Elbow Curve (UNSCALED)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Inertia")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2. Silhouette Score Comparison
df_sil_scaled = silhouette_by_k(X_train_scaled)
best_k_sil_scaled = df_sil_scaled.loc[df_sil_scaled["silhouette"].idxmax(), "k"]

df_sil_unscaled = silhouette_by_k(X_train_unscaled)
best_k_sil_unscaled = df_sil_unscaled.loc[df_sil_unscaled["silhouette"].idxmax(), "k"]

fig, axes = plt.subplots(1, 2)
axes[0].plot(df_sil_scaled["k"], df_sil_scaled["silhouette"], marker="s", color="green", linestyle="-")
axes[0].axvline(best_k_sil_scaled, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_scaled}")
axes[0].set_title("Silhouette Curve (SCALED)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Average Silhouette Score")
axes[0].legend()

axes[1].plot(df_sil_unscaled["k"], df_sil_unscaled["silhouette"], marker="s", color="purple", linestyle="-")
axes[1].axvline(best_k_sil_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_unscaled}")
axes[1].set_title("Silhouette Curve (UNSCALED)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Average Silhouette Score")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary Tables
print("=== SCALED SUMMARY ===")
summary_scaled = summarize_k_selection(df_elbow_scaled, df_sil_scaled)
display(summary_scaled)

print("\n=== UNSCALED SUMMARY ===")
summary_unscaled = summarize_k_selection(df_elbow_unscaled, df_sil_unscaled)
display(summary_unscaled)

In [ ]:
# Based on the curves, choose the best K for both branches
CHOSEN_K_SCALED = max(best_k_elbow_scaled, best_k_sil_scaled)
CHOSEN_K_UNSCALED = max(best_k_elbow_unscaled, best_k_sil_unscaled)

print(f"Applying SCALED K-Means with K = {CHOSEN_K_SCALED}")
print(f"Applying UNSCALED K-Means with K = {CHOSEN_K_UNSCALED}")

labels_scaled, _, model_scaled = run_kmeans(X_train_scaled, k=CHOSEN_K_SCALED)
labels_unscaled, _, model_unscaled = run_kmeans(X_train_unscaled, k=CHOSEN_K_UNSCALED)

# Save labels
pd.DataFrame({"cluster_id": labels_scaled}).to_csv("data/processed/labels_scaled.csv", index=False)
pd.DataFrame({"cluster_id": labels_unscaled}).to_csv("data/processed/labels_unscaled.csv", index=False)
print("Saved both labels arrays to data/processed/")

In [ ]:
# Describe Centroids physically
# Note: we drop IDs from X_unscaled as well for description
X_unscaled_features = X_unscaled.drop(columns=cols_to_drop)

print("=== SCALED CENTROIDS (in original units) ===")
centroids_physical_scaled = describe_centroids(model_scaled, X_unscaled_features, labels_scaled)
display(centroids_physical_scaled)

print("\n=== UNSCALED CENTROIDS ===")
centroids_physical_unscaled = describe_centroids(model_unscaled, X_unscaled_features, labels_unscaled)
display(centroids_physical_unscaled)

In [ ]:
# Plotting the clusters geographically side-by-side
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Draw pitch bounds and goal line for SCALED
axes[0].plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
axes[0].plot([120, 120], [36, 44], color="red", linewidth=4)
sns.scatterplot(
    ax=axes[0], x=X_unscaled["location_x"], y=X_unscaled["location_y"], 
    hue=labels_scaled, palette="tab10", alpha=0.6, s=20
)
axes[0].set_title(f"SCALED Shot Zones (K={CHOSEN_K_SCALED})")
axes[0].legend(title="Cluster")

# Draw pitch bounds and goal line for UNSCALED
axes[1].plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
axes[1].plot([120, 120], [36, 44], color="red", linewidth=4)
sns.scatterplot(
    ax=axes[1], x=X_unscaled["location_x"], y=X_unscaled["location_y"], 
    hue=labels_unscaled, palette="tab10", alpha=0.6, s=20
)
axes[1].set_title(f"UNSCALED Shot Zones (K={CHOSEN_K_UNSCALED})")
axes[1].legend(title="Cluster")

plt.show()